In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import wandb

In [2]:
def get_color_cycle(algorithms):
  """
  Creates a circular buffer of colors for plotting multiple algorithms.

  Args:
    algorithms: A list of algorithm names.

  Returns:
    A dictionary mapping each algorithm to a unique color.
  """
  colors = itertools.cycle(plt.cm.get_cmap('tab10').colors)  # You can choose a different colormap
  color_map = {}
  for algorithm in algorithms:
    color_map[algorithm] = next(colors)
  return color_map



def add_ema_columns(df, alpha=0.5, adjust=True):
  """
  Adds EMA columns for columns ending with "_reward" or "_budget".

  Args:
    df: The pandas DataFrame.
    alpha: The smoothing factor for the EMA calculation (default: 0.5).

  Returns:
    The DataFrame with added EMA columns.
  """
  for column in df.columns:
    if column.endswith("_reward") or column.endswith("_budget"):
      new_column_name = column + "_ema"
      df[new_column_name] = df[column].ewm(alpha=alpha, adjust=adjust).mean()
  return df


def add_ra_columns(df, num_avg_samples=10):
  """
  Adds Running AVG columns for columns ending with "_reward" or "_budget".

  Args:
    df: The pandas DataFrame.
    num_avg_samples: The number of samples to consider for the moving average.

  Returns:
    The DataFrame with added EMA columns.
  """
  for column in df.columns:
    if column.endswith("_reward") or column.endswith("_budget"):
      print('creating running average of column', column)
      new_column_name = column + "_ra"
      df[new_column_name] = df[column].rolling(window=num_avg_samples, min_periods=1).mean()
  return df


# Create a custom handler for vertical lines in legend
class VerticalLineHandler:
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        x0, y0 = handlebox.xdescent, handlebox.ydescent
        width = handlebox.width
        height = handlebox.height
        # Create a vertical line in the center of the box
        line = plt.Line2D([x0 + width/2, x0 + width/2],
                         [y0, y0 + height],
                         color=orig_handle.get_color(),
                         linestyle=orig_handle.get_linestyle())
        handlebox.add_artist(line)
        return line


# formatting stuff:
title_font_dict = {'weight': 'bold', 'size': 30}
axis_font_dict = {'weight': 'bold', 'size': 18}
legend_font_dict = {'weight': 'bold', 'size': 16}

In [ ]:


# Create the plot with specified dimensions
fig, ax = plt.subplots(figsize=(20, 3))  # Adjust width and height as needed

# Loop through each algorithm and plot the data
for algorithm in algorithms:
    # Plot the "_reward" column as a line
    print(f'now plotting: {algorithm}_reward_ra')
    ax.plot(omni['Step'], omni[f'{algorithm}_reward_ra'], label=f'{algorithm}', alpha=0.75, color=color_map[algorithm], lw=2)

    # Plot crosses where the "_es" column has a 1
    es_indices = omni.index[omni[f'{algorithm}_es'] == 1].tolist()
    es_rewards = omni.loc[es_indices, f'{algorithm}_reward_ra'].tolist()
    ax.plot(omni.loc[es_indices,'Step'], es_rewards, 'x', markersize=6, markeredgewidth=3, alpha=0.95, color=color_map[algorithm])

ax.plot([], [], 'x', markersize=8, markeredgewidth=4, color='black', label='Episode Restart')

# Customize the plot
ax.set_xlim(left=40)  # You can replace 0 with your desired xmin value

ax.set_xlabel('TRAINING STEPS', fontdict=axis_font_dict)
ax.set_ylabel('REWARD', fontdict=axis_font_dict)
ax.set_title('REWARDS OVER TIME', fontdict=title_font_dict, y=0.6, x=0.8)
ax.legend(prop=legend_font_dict, bbox_to_anchor=(0.9, 0.5))

ax.xaxis.set_label_coords(0.6, 0.1)  # Moves xlabel right and down
ax.yaxis.set_label_coords(0.02, 0.7)  # Moves ylabel left

# Increase font size of xticks and yticks
plt.xticks(fontsize=18, weight='bold')
plt.yticks(fontsize=18, weight='bold')

# Display the plot
plt.show()

In [3]:
def prepend_to_columns(df, string):
  """Prepends a string followed by an underscore to all column names in a DataFrame.

  Args:
    df: The pandas DataFrame to modify.
    string: The string to prepend to each column name.

  Returns:
    A new DataFrame with modified column names.
  """
  new_columns = [(string + '_' + col if col != 'step' else 'step') for col in df.columns ]

  return df.rename(columns=dict(zip(df.columns, new_columns)))


def fill_ending_episode(df):
    # Get the index positions of all '1's
    ones_indices = df.index[df['ending_episode'] == 1].tolist()

    # Initialize a list to hold the counts
    counts = []
    prev_one_index = 0

    for i in range(len(df)):
        # If current index is in ones_indices, append 0 (or keep it as is)
        if i in ones_indices:
            if len(counts) > 0:
                counts.append(counts[-1])
            else:
                counts.append(0)
            prev_one_index = i
        else:
            # Find the next '1' index
            next_one_index = next((index for index in ones_indices if index > i), None)
            if next_one_index is not None:
                counts.append(next_one_index - prev_one_index)
            else:
                counts.append(0)  # No more '1's found

    df['episode_length'] = counts
    return df


In [49]:
api = wandb.Api()

In [50]:
runs = api.runs("jfcevallos/TIGER2.0")

In [51]:
default_run_id = "kp5mp4f9"

for run in runs:
    if run.id == default_run_id:
        print(run.name, run.id)
        default_run_name = run.name
        default_run = run
        # print('scanning run history:')
        # default_run_history = list(run.history(samples=10000))    
        # print('creating dataframe:')
        # default_run_history = pd.DataFrame(history_rows)
        default_run_history = run.history(samples=20000) 
        break

DDQN kp5mp4f9


# ASAP Architectures

In [ ]:
asap_runs = {}
for run in runs:
    if run.group == "arch_agency":
        print(run.name, run.id)
        asap_runs[run.name] = {
            "id": run.id}
        # print('scanning run history:')
        # history_rows = list(run.scan_history(keys=None))    
        # print('creating dataframe:')
        # asap_runs[run.name]["history"] = pd.DataFrame(history_rows)
        asap_runs[run.name]["history"] = run.history(samples=20000)
        
    
asap_runs["agency_optim"]["display_name"] = "Optimised Models"
asap_runs["agency_simple_krloss"]["display_name"] = "Simple_KRLoss"
asap_runs["agency_dotprod_kr"]["display_name"] = "DotProd_KR"
asap_runs["agency_dotprod_kr_simplekrloss"]["display_name"] = "DotProd_KR_SimpleKRLoss"

asap_runs[default_run_name] = {
    "id": default_run_id,
    "history": default_run_history,
    "display_name": "Default"
}

agency_simple_krloss oa9to1ej
agency_optim 3ctewnbd
agency_dotprod_kr ca2z4r0q
agency_dotprod_kr_simplekrloss u273eepm


In [56]:
[col for col in list(default_run_history.columns) if 'AGENT' in col]

['AGENT/bad_classification_cost',
 'AGENT/clustering_reward',
 'AGENT/budget',
 'AGENT/generic_reward',
 'AGENT/epistemic_costs',
 'AGENT/known traffic action',
 'AGENT/Epistemic Actions taken',
 'AGENT/no_confidence_penalty',
 'AGENT/rewards_per_accepted_clusters',
 'AGENT/classification_reward',
 'AGENT/rewards_per_blocked_clusters',
 'AGENT/correct_classification_rewards']

In [ ]:
for run_name, run_dict in asap_runs.items():
    
    reward_series = run_dict["history"]["AGENT/generic_reward"]


agency_simple_krloss 18809
agency_optim 14854
agency_dotprod_kr 17833
agency_dotprod_kr_simplekrloss 67000
